# LeetCode 1764 — Form Array by Concatenating Subarrays of Another Array bằng Python

Link đề: https://leetcode.com/problems/form-array-by-concatenating-subarrays-of-another-array/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Greedy, String Matching

Skeleton LeetCode:

```python
class Solution:
    def canChoose(self, groups: List[List[int]], nums: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu ba ràng buộc: các mảng con phải **bằng** `groups[i]`, **rời nhau**, và **đúng thứ tự**.
2. Viết vét cạn (quay lui thử mọi vị trí bắt đầu) làm đáp án chuẩn.
3. Chứng minh tham lam **"khớp sớm nhất có thể"** là tối ưu bằng lập luận hoán đổi.
4. Cài so khớp bằng slicing, bằng so sánh từng phần tử, và bằng **KMP** để đạt `O(m + Σ|g|)`.
5. Kiểm thử với các trường hợp chồng lấn, sai thứ tự, và mảng con lặp lại.

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `groups` gồm `n` mảng và mảng `nums`. Hỏi có thể chọn `n` **mảng con** (liên tiếp) **rời nhau** của `nums` sao cho mảng con thứ `i` **bằng** `groups[i]`, và mảng con thứ `i−1` nằm **trước** mảng con thứ `i` trong `nums` (cùng thứ tự với `groups`) hay không.

Rời nhau: không có chỉ số `k` nào mà `nums[k]` thuộc hai mảng con.

**Ví dụ 1:** `groups = [[1,-1,-1],[3,-2,0]]`, `nums = [1,-1,0,1,-1,-1,3,-2,0]` → `True` (chọn `nums[3..5]` và `nums[6..8]`).

**Ví dụ 2:** `groups = [[10,-2],[1,2,3,4]]`, `nums = [1,2,3,4,10,-2]` → `False` (`[10,-2]` phải đứng trước `[1,2,3,4]`).

**Ví dụ 3:** `groups = [[1,2,3],[3,4]]`, `nums = [7,7,1,2,3,4,7,7]` → `False` (hai mảng con buộc phải dùng chung `nums[4] = 3`).

**Ràng buộc:** `1 ≤ n ≤ 10³`; `1 ≤ groups[i].length`, `Σ groups[i].length ≤ 10³`; `1 ≤ nums.length ≤ 10³`; `−10⁷ ≤ giá trị ≤ 10⁷`.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [
    ([[1, -1, -1], [3, -2, 0]], [1, -1, 0, 1, -1, -1, 3, -2, 0], True),
    ([[10, -2], [1, 2, 3, 4]], [1, 2, 3, 4, 10, -2], False),
    ([[1, 2, 3], [3, 4]], [7, 7, 1, 2, 3, 4, 7, 7], False),
]
for g, nums, exp in examples:
    print(f"groups={g}, nums={nums} -> kỳ vọng {exp}")

groups=[[1, -1, -1], [3, -2, 0]], nums=[1, -1, 0, 1, -1, -1, 3, -2, 0] -> kỳ vọng True
groups=[[10, -2], [1, 2, 3, 4]], nums=[1, 2, 3, 4, 10, -2] -> kỳ vọng False
groups=[[1, 2, 3], [3, 4]], nums=[7, 7, 1, 2, 3, 4, 7, 7] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 So sánh lát cắt list

`nums[j:j+L] == g` so sánh từng phần tử, `O(L)`. Nếu `j + L > len(nums)` lát cắt ngắn hơn `g` nên tự động khác.

In [2]:
nums = [1, -1, 0, 1, -1, -1]
print(nums[3:6] == [1, -1, -1], nums[0:3] == [1, -1, -1], nums[5:8], nums[5:8] == [-1, 0, 1])

True False [-1] False


### 3.2 Tìm vị trí xuất hiện đầu tiên của mảng con từ vị trí `start`

Duyệt `j = start, start+1, …` tới khi `nums[j:j+L] == g`. Đây là "tìm chuỗi con" thô — `O((m − L) · L)`.

In [3]:
def find_from(nums, g, start):
    L = len(g)
    for j in range(start, len(nums) - L + 1):
        if nums[j:j + L] == g:
            return j
    return -1

print(find_from([1, -1, 0, 1, -1, -1, 3], [1, -1, -1], 0), find_from([1, 2, 3], [4], 0))

3 -1


### 3.3 Mảng tiền tố–hậu tố (failure function) của KMP

`lps[k]` = độ dài tiền tố thật dài nhất của `g[:k+1]` đồng thời là hậu tố của nó. Nhờ đó khi so khớp bị lệch, ta không lùi con trỏ trên `nums` ⇒ tìm kiếm `O(m + L)`.

In [4]:
def build_lps(g):
    lps = [0] * len(g); k = 0
    for i in range(1, len(g)):
        while k and g[i] != g[k]:
            k = lps[k - 1]
        if g[i] == g[k]:
            k += 1
        lps[i] = k
    return lps

print(build_lps([1, 2, 1, 2, 3]), build_lps([5, 5, 5]), build_lps([1, 2, 3]))

[0, 0, 1, 2, 0] [0, 1, 2] [0, 0, 0]


### 3.4 Quay lui với ghi nhớ

`can(i, j)` = có thể đặt `groups[i:]` vào `nums[j:]` không. Thử mọi vị trí bắt đầu `p ≥ j` của `groups[i]`; nếu khớp thì gọi `can(i + 1, p + len)`. Ghi nhớ theo `(i, j)` giới hạn số trạng thái `O(n · m)`.

## 4. Ý tưởng nền tảng / trực giác

Làm tay **Ví dụ 3**: `groups = [[1,2,3],[3,4]]`, `nums = [7,7,1,2,3,4,7,7]`.
- `[1,2,3]` chỉ xuất hiện ở vị trí 2 (chiếm `nums[2..4]`).
- `[3,4]` chỉ xuất hiện ở vị trí 4 — nhưng `nums[4]` đã bị dùng. Không còn chỗ nào khác ⇒ `False`.

Làm tay **Ví dụ 1**: `[1,-1,-1]` thử vị trí 0: `[1,-1,0]` ≠; vị trí 3: khớp ⇒ con trỏ nhảy tới 6. `[3,-2,0]` ở vị trí 6 khớp ⇒ `True`.

**Trực giác tham lam:** với nhóm hiện tại, chọn chỗ khớp **sớm nhất** — vì nó kết thúc sớm nhất, để lại **nhiều chỗ nhất** cho các nhóm sau.

In [5]:
g, nums, _ = examples[2]
for i, grp in enumerate(g):
    print(grp, "xuất hiện tại:", [j for j in range(len(nums) - len(grp) + 1) if nums[j:j + len(grp)] == grp])

[1, 2, 3] xuất hiện tại: [2]
[3, 4] xuất hiện tại: [4]


## 5. Cách 1 — Brute Force / baseline: quay lui thử mọi vị trí

In [6]:
def can_bruteforce(groups, nums):
    n, m = len(groups), len(nums)
    @lru_cache(maxsize=None)
    def can(i, j):
        if i == n:
            return True
        g = groups[i]; L = len(g)
        for p in range(j, m - L + 1):
            if nums[p:p + L] == g and can(i + 1, p + L):
                return True
        return False
    return can(0, 0)

for g, nums, exp in examples:
    got = can_bruteforce(g, nums); print(got, "kỳ vọng", exp); assert got == exp

True kỳ vọng True
False kỳ vọng False
False kỳ vọng False


**Độ phức tạp:** `O(n · m)` trạng thái × `O(m · L)` mỗi trạng thái ⇒ có thể tới `O(n · m² · L)` — chấp nhận được cho input nhỏ nhưng thừa thãi. Đệ quy sâu tới `n = 1000` cũng chạm giới hạn đệ quy mặc định.

**Nút thắt:** ta thử **mọi** vị trí cho mỗi nhóm, kể cả các vị trí muộn hơn vị trí sớm nhất — trong khi vị trí sớm nhất luôn tốt nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** đặt `groups[0]` ở lần xuất hiện **sớm nhất** trong `nums`, rồi `groups[1]` ở lần xuất hiện sớm nhất **sau** đó, …

**Bổ đề hoán đổi:** nếu tồn tại một cách đặt hợp lệ trong đó `groups[i]` bắt đầu ở `p`, và lần xuất hiện sớm nhất (sau phần đã dùng) là `p* ≤ p`, thì dời `groups[i]` về `p*` vẫn hợp lệ: nó kết thúc ở `p* + L ≤ p + L`, không đè lên nhóm trước (vì `p*` ≥ điểm kết thúc nhóm trước), và để lại cho các nhóm sau **ít nhất** nhiều chỗ như cũ.

**Quy tắc:**
- **Nếu** `nums[j:j+L] == groups[i]` **thì** nhận, `j += L`, `i += 1`.
- **Ngược lại** `j += 1`.
- Trả `True` khi đã nhận hết `n` nhóm.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam + so sánh lát cắt

In [7]:
def can_greedy_slice(groups, nums):
    i = j = 0
    n, m = len(groups), len(nums)
    while i < n and j < m:
        g = groups[i]
        if nums[j:j + len(g)] == g:
            j += len(g); i += 1
        else:
            j += 1
    return i == n

print([can_greedy_slice(g, nums) for g, nums, _ in examples])

[True, False, False]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** mỗi vị trí `j` có thể so tới `L_i` phần tử ⇒ `O(m · max L)` tệ nhất, tức `≤ 10⁶`.

### 7.2 Cách 3 — Tham lam + so sánh từng phần tử (dừng sớm, không tạo lát cắt)

In [8]:
class SolutionPointers:
    def canChoose(self, groups: List[List[int]], nums: List[int]) -> bool:
        j, m = 0, len(nums)
        for g in groups:
            L = len(g)
            while j + L <= m:
                k = 0
                while k < L and nums[j + k] == g[k]:
                    k += 1
                if k == L:
                    break
                j += 1
            else:
                return False                 # không tìm thấy g trong phần còn lại
            j += L
        return True

print([SolutionPointers().canChoose(g, nums) for g, nums, _ in examples])

[True, False, False]


Cú pháp `while … else`: khối `else` chạy khi vòng `while` kết thúc **không** do `break` — tức là đã hết chỗ mà chưa khớp.

**Độ phức tạp Cách 3:** như Cách 2, `O(m · max L)`, Space `O(1)`.

### 7.3 Cách 4 — Tham lam + KMP: `O(m + Σ L)`

Với mỗi nhóm, chạy KMP tìm lần xuất hiện đầu tiên trong `nums[j:]`. Con trỏ trên `nums` không bao giờ lùi ⇒ tổng các lần tìm là `O(m)` cộng `O(Σ L)` để dựng `lps`.

In [9]:
def kmp_find(nums, g, start):
    lps = build_lps(g); k = 0
    for j in range(start, len(nums)):
        while k and nums[j] != g[k]:
            k = lps[k - 1]
        if nums[j] == g[k]:
            k += 1
            if k == len(g):
                return j - len(g) + 1
    return -1

def can_kmp(groups, nums):
    j = 0
    for g in groups:
        p = kmp_find(nums, g, j)
        if p == -1:
            return False
        j = p + len(g)
    return True

print([can_kmp(g, nums) for g, nums, _ in examples])
print("kmp_find:", kmp_find([1, 2, 1, 2, 1, 2, 3], [1, 2, 1, 2, 3], 0))

[True, False, False]
kmp_find: 2


**Code nộp LeetCode (Cách 4):**

In [10]:
class SolutionKMP:
    def canChoose(self, groups: List[List[int]], nums: List[int]) -> bool:
        j = 0
        for g in groups:
            lps = [0] * len(g); k = 0
            for i in range(1, len(g)):
                while k and g[i] != g[k]:
                    k = lps[k - 1]
                if g[i] == g[k]:
                    k += 1
                lps[i] = k
            k = 0; found = -1
            while j < len(nums):
                while k and nums[j] != g[k]:
                    k = lps[k - 1]
                if nums[j] == g[k]:
                    k += 1
                j += 1
                if k == len(g):
                    found = j
                    break
            if found == -1:
                return False
        return True

print([SolutionKMP().canChoose(g, nums) for g, nums, _ in examples])

[True, False, False]


Ở bản nộp, `j` sau khi khớp đã trỏ đúng **ngay sau** mảng con vừa tìm, nên nhóm tiếp theo bắt đầu tìm từ đó — đảm bảo rời nhau.

**Độ phức tạp Cách 4:** Time `O(m + Σ L)` vì `j` chỉ tăng, Space `O(max L)`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1**:

| bước | i (nhóm) | j | nums[j:j+L] | khớp? | hành động |
|---|---|---|---|---|---|
| 1 | 0 `[1,-1,-1]` | 0 | `[1,-1,0]` | không | j=1 |
| 2 | 0 | 1 | `[-1,0,1]` | không | j=2 |
| 3 | 0 | 2 | `[0,1,-1]` | không | j=3 |
| 4 | 0 | 3 | `[1,-1,-1]` | **có** | j=6, i=1 |
| 5 | 1 `[3,-2,0]` | 6 | `[3,-2,0]` | **có** | j=9, i=2 |

In [11]:
def trace_greedy(groups, nums):
    i = j = 0; step = 0
    while i < len(groups) and j < len(nums):
        step += 1
        g = groups[i]; window = nums[j:j + len(g)]
        ok = window == g
        print(f"bước {step}: nhóm {i} {g}, j={j}, cửa sổ={window} -> {'KHỚP' if ok else 'không'}")
        if ok:
            j += len(g); i += 1
        else:
            j += 1
    print(f"đã đặt {i}/{len(groups)} nhóm ->", i == len(groups))
    return i == len(groups)

assert trace_greedy(*examples[0][:2]) is True
print()
assert trace_greedy(*examples[2][:2]) is False

bước 1: nhóm 0 [1, -1, -1], j=0, cửa sổ=[1, -1, 0] -> không
bước 2: nhóm 0 [1, -1, -1], j=1, cửa sổ=[-1, 0, 1] -> không
bước 3: nhóm 0 [1, -1, -1], j=2, cửa sổ=[0, 1, -1] -> không
bước 4: nhóm 0 [1, -1, -1], j=3, cửa sổ=[1, -1, -1] -> KHỚP
bước 5: nhóm 1 [3, -2, 0], j=6, cửa sổ=[3, -2, 0] -> KHỚP
đã đặt 2/2 nhóm -> True

bước 1: nhóm 0 [1, 2, 3], j=0, cửa sổ=[7, 7, 1] -> không
bước 2: nhóm 0 [1, 2, 3], j=1, cửa sổ=[7, 1, 2] -> không
bước 3: nhóm 0 [1, 2, 3], j=2, cửa sổ=[1, 2, 3] -> KHỚP
bước 4: nhóm 1 [3, 4], j=5, cửa sổ=[4, 7] -> không
bước 5: nhóm 1 [3, 4], j=6, cửa sổ=[7, 7] -> không
bước 6: nhóm 1 [3, 4], j=7, cửa sổ=[7] -> không
đã đặt 1/2 nhóm -> False


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — tham lam + so lát cắt (ngắn, đủ nhanh với giới hạn `10³`).

In [12]:
class Solution:
    def canChoose(self, groups: List[List[int]], nums: List[int]) -> bool:
        i = j = 0
        n, m = len(groups), len(nums)
        while i < n and j < m:
            g = groups[i]
            if nums[j:j + len(g)] == g:
                j += len(g)
                i += 1
            else:
                j += 1
        return i == n

for g, nums, exp in examples:
    got = Solution().canChoose(g, nums); print(got, exp); assert got == exp

True True
False False
False False


## 10. Vì sao thuật toán đúng

**(⇒) Tham lam trả `True` thì có lời giải:** các mảng con nó chọn bằng đúng `groups[i]`, rời nhau (mỗi lần nhận, `j` nhảy qua hết mảng con vừa chọn) và đúng thứ tự (`j` chỉ tăng).

**(⇐) Có lời giải thì tham lam trả `True`:** gọi lời giải bất kỳ có vị trí bắt đầu `q_0 < q_1 < …`. Gọi `p_i` là vị trí tham lam chọn. Chứng minh quy nạp `p_i ≤ q_i`:
- `i = 0`: `p_0` là lần xuất hiện sớm nhất của `groups[0]` ⇒ `p_0 ≤ q_0`.
- Giả sử `p_{i−1} ≤ q_{i−1}`. Lời giải có `q_i ≥ q_{i−1} + L_{i−1} ≥ p_{i−1} + L_{i−1}` = điểm tham lam bắt đầu tìm nhóm `i`. Vậy `q_i` là một vị trí khớp hợp lệ trong vùng tham lam đang tìm ⇒ tham lam tìm thấy vị trí sớm nhất `p_i ≤ q_i`.

Vì vậy tham lam đặt được mọi nhóm. ∎

**Tham lam duyệt `j += 1` không bỏ sót:** mọi vị trí giữa điểm bắt đầu tìm và `p_i` đều đã được thử và không khớp.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(nums)`, `S = Σ len(groups[i])`, `Lmax = max len(groups[i])`.

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui + ghi nhớ | `O(n · m · m · Lmax)` tệ nhất | `O(n · m)` |
| 2. Tham lam + slicing | `O(m · Lmax + S)` | `O(Lmax)` (lát cắt tạm) |
| 3. Tham lam + so từng phần tử | `O(m · Lmax + S)` | `O(1)` |
| 4. Tham lam + KMP | `O(m + S)` | `O(Lmax)` |

Trường hợp tệ cho Cách 2/3: `nums` toàn `0`, nhóm là `[0,0,…,0,1]` — mỗi vị trí phải so gần hết nhóm.

In [13]:
nums = [0] * 1000; groups = [[0] * 499 + [1]]
for name, f in [("Cách 2", Solution().canChoose), ("Cách 3", SolutionPointers().canChoose), ("Cách 4", SolutionKMP().canChoose)]:
    t = time.perf_counter(); r = f(groups, nums); dt = time.perf_counter() - t
    print(f"{name}: {r} ({dt*1000:.1f} ms)")

Cách 2: False (0.5 ms)
Cách 3: False (10.1 ms)
Cách 4: False (0.2 ms)


## 12. Bộ test

In [14]:
ALL = {"slice": can_greedy_slice, "kmp": can_kmp, "SolPointers": SolutionPointers().canChoose,
       "SolKMP": SolutionKMP().canChoose, "submit": Solution().canChoose}
def run_all(groups, nums):
    res = {k: f(groups, nums) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (groups, nums, res)
    return next(iter(res.values()))

for g, nums, exp in examples:
    assert run_all(g, nums) == exp
edge = [([[1]], [1], True), ([[1]], [2], False), ([[1], [1]], [1], False), ([[1], [1]], [1, 1], True),
        ([[1, 2]], [1], False), ([[1, 1], [1, 1]], [1, 1, 1], False), ([[1, 1], [1, 1]], [1, 1, 1, 1], True),
        ([[1, 2, 1], [2, 1]], [1, 2, 1, 2, 1], True)]
for g, nums, exp in edge:
    assert can_bruteforce(g, nums) == exp and run_all(g, nums) == exp, (g, nums)
rng = random.Random(1764)
cnt = {True: 0, False: 0}
for _ in range(2000):
    nums = [rng.randint(0, 2) for _ in range(rng.randint(1, 14))]
    groups = [[rng.randint(0, 2) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 4))]
    if rng.random() < 0.4:                                   # tạo trường hợp True có chủ đích
        groups = []; p = 0
        while p < len(nums) and len(groups) < 4:
            p += rng.randint(0, 2); L = rng.randint(1, 3)
            if p + L <= len(nums): groups.append(nums[p:p + L])
            p += L
        groups = groups or [[nums[0]]]
    r = run_all(groups, nums); assert r == can_bruteforce(groups, nums), (groups, nums); cnt[r] += 1
print("Ví dụ, edge case, 2000 test ngẫu nhiên OK | phân bố:", cnt)

Ví dụ, edge case, 2000 test ngẫu nhiên OK | phân bố: {True: 1044, False: 956}


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra từng nhóm có xuất hiện trong `nums` (bỏ qua thứ tự & rời nhau)

In [15]:
def wrong_any_position(groups, nums):
    return all(any(nums[j:j + len(g)] == g for j in range(len(nums))) for g in groups)

for g, nums, exp in examples[1:]:
    print("sai :", wrong_any_position(g, nums), "| đúng:", exp)

sai : True | đúng: False
sai : True | đúng: False


**Vì sao sai:** Ví dụ 2 vi phạm thứ tự, Ví dụ 3 vi phạm tính rời nhau — nhưng mỗi nhóm đều "có xuất hiện". **Sửa:** tìm nhóm sau **bắt đầu từ sau** nhóm trước.

### Lỗi 2 — Sau khi khớp chỉ nhảy `j += 1` (cho phép chồng lấn)

In [16]:
def wrong_overlap(groups, nums):
    i = j = 0
    while i < len(groups) and j < len(nums):
        g = groups[i]
        if nums[j:j + len(g)] == g:
            i += 1; j += 1                                # SAI: phải nhảy len(g)
        else:
            j += 1
    return i == len(groups)

g, nums, exp = examples[2]
print("sai :", wrong_overlap(g, nums), "| đúng:", exp)

sai : True | đúng: False


**Vì sao sai:** nhảy 1 khiến nhóm sau có thể dùng lại phần tử của nhóm trước. **Sửa:** `j += len(g)`.

### Lỗi 3 — Gặp lệch thì nhảy qua cả cửa sổ (`j += len(g)`)

In [17]:
def wrong_skip_window(groups, nums):
    i = j = 0
    while i < len(groups) and j < len(nums):
        g = groups[i]
        if nums[j:j + len(g)] == g:
            j += len(g); i += 1
        else:
            j += len(g)                                    # SAI: bỏ sót vị trí khớp bên trong cửa sổ
    return i == len(groups)

g, nums, exp = examples[0]
print("sai :", wrong_skip_window(g, nums), "| đúng:", exp)

sai : True | đúng: True


**Vì sao sai:** chỗ khớp có thể bắt đầu ở **giữa** cửa sổ vừa thử. Ở Ví dụ 1 code sai tình cờ vẫn ra đúng, vì vị trí khớp (3) trùng với một bước nhảy `0 → 3`. Chỉ cần lệch một vị trí là lộ lỗi:

In [18]:
g, nums = [[1, 2]], [1, 1, 2]            # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_skip_window(g, nums), "| đúng:", Solution().canChoose(g, nums), "| brute:", can_bruteforce(g, nums))

sai : False | đúng: True | brute: True


**Sửa:** khi lệch chỉ tiến `j += 1` (hoặc dùng KMP để nhảy **an toàn** theo `lps`).

### Lỗi 4 — Dùng chuỗi: `",".join` rồi `str.find` mà quên dấu phân cách đầu/cuối

In [19]:
def wrong_string_find(groups, nums):
    s = ",".join(map(str, nums)); pos = 0
    for g in groups:
        t = ",".join(map(str, g))
        k = s.find(t, pos)
        if k == -1:
            return False
        pos = k + len(t)
    return True

g, nums = [[2]], [12]                     # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_string_find(g, nums), "| đúng:", Solution().canChoose(g, nums))

sai : True | đúng: False


**Vì sao sai:** `"2"` là chuỗi con của `"12"` dù `2` không phải phần tử của `nums`. **Sửa:** bao mỗi số bằng dấu phân cách ở **cả hai** phía (ví dụ `",12,"` và `",2,"`) hoặc so sánh trên list.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đặt các khối theo **thứ tự**, **rời nhau**" ⇒ tham lam **sớm nhất có thể** (giống khớp dãy con).
- So khớp mảng con lặp lại ⇒ cân nhắc **KMP** / Z-function / hashing khi giới hạn lớn.
- Khi lệch: tiến **1** (hoặc theo `lps`), không nhảy cả cửa sổ.
- Chuyển sang chuỗi để dùng `find` phải có **dấu phân cách hai phía**.

## 15. Biến thể của bài

1. **Cho phép đổi thứ tự các nhóm**: bài khó hơn hẳn (NP-khó dạng tổng quát); với `n` nhỏ dùng bitmask DP.
2. **Đếm số cách chọn** các vị trí: quy hoạch động `dp[i][j]`.
3. **392 Is Subsequence**: phiên bản mỗi nhóm có độ dài 1.
4. **28 Find the Index of the First Occurrence in a String**: KMP cơ bản.
5. **Trả về vị trí bắt đầu** của từng nhóm (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Thử mọi vị trí | lớn | `O(nm)` | Kiểm chứng |
| 2. Tham lam + slicing | Khớp sớm nhất | `O(m·Lmax + S)` | `O(Lmax)` | **Bản nộp** |
| 3. Tham lam + so từng phần tử | Như 2, không cấp phát | `O(m·Lmax + S)` | `O(1)` | Ngôn ngữ khác |
| 4. Tham lam + KMP | Không lùi con trỏ | `O(m + S)` | `O(Lmax)` | Giới hạn lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([[1,2],[2,3]], [1,2,3])`; (b) `([[1,2],[2,3]], [1,2,2,3])`; (c) `([[5]], [5,5,5])`; (d) `([[1],[2],[1]], [1,2])`.

**Bài 2 (tính tay):** Trace tham lam cho `groups = [[2,1],[1,2]]`, `nums = [2,2,1,2,1,2]`.

**Bài 3 (điền TODO):** Hoàn thiện `first_match(nums, g, start)` trả vị trí khớp đầu tiên `≥ start` hoặc `−1`.

**Bài 4 (tự viết lại):** Viết `positions(groups, nums)` trả list vị trí bắt đầu của mỗi nhóm theo tham lam (hoặc `None`), và kiểm tra tính hợp lệ.

**Bài 5 (mini challenge):** Viết `max_groups(groups, nums)` trả **số nhóm đầu tiên nhiều nhất** (`groups[0..k−1]`) có thể đặt được.

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def first_match_todo(nums, g, start):
    # TODO: for j in range(start, len(nums) - len(g) + 1): nếu nums[j:j+len(g)] == g thì trả j
    return -1

In [21]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def positions(groups, nums):
    # TODO: dùng first_match lần lượt; nếu -1 thì trả None
    pass

def max_groups(groups, nums):
    # TODO: như tham lam, đếm i khi dừng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm nhóm 0 sớm nhất. Mức 2: nhóm sau tìm từ sau nhóm trước. Mức 3: (a) `False` (chung số 2); (b) `True`; (c) `True`; (d) `False`.

**Bài 2.** Mức 1: `[2,1]` ở vị trí 0? `[2,2]` không; vị trí 1 `[2,1]` khớp ⇒ `j = 3`. Mức 2: `[1,2]` ở vị trí 3? `[2,1]` không; vị trí 4 `[1,2]` khớp. Mức 3: `True`, vị trí `[1, 4]`.

**Bài 3.** Mức 1: giới hạn `j + len(g) ≤ len(nums)`. Mức 2: so lát cắt. Mức 3: hết vòng trả `−1`.

**Bài 4.** Mức 1: `j = 0`. Mức 2: `p = first_match(nums, g, j)`, `j = p + len(g)`. Mức 3: kiểm tra `nums[p:p+L] == g` và các đoạn không chồng nhau.

**Bài 5.** Mức 1: tham lam dừng khi không khớp được nữa. Mức 2: trả `i`. Mức 3: vì tham lam tối ưu cho mọi tiền tố của `groups`, `i` chính là đáp án; đối chiếu brute force thử từng `k`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
b1 = [Solution().canChoose(g, n) for g, n in [([[1, 2], [2, 3]], [1, 2, 3]), ([[1, 2], [2, 3]], [1, 2, 2, 3]), ([[5]], [5, 5, 5]), ([[1], [2], [1]], [1, 2])]]
print("Bài 1:", b1); assert b1 == [False, True, True, False]
print("Bài 2:"); assert trace_greedy([[2, 1], [1, 2]], [2, 2, 1, 2, 1, 2]) is True

Bài 1: [False, True, True, False]
Bài 2:
bước 1: nhóm 0 [2, 1], j=0, cửa sổ=[2, 2] -> không
bước 2: nhóm 0 [2, 1], j=1, cửa sổ=[2, 1] -> KHỚP
bước 3: nhóm 1 [1, 2], j=3, cửa sổ=[2, 1] -> không
bước 4: nhóm 1 [1, 2], j=4, cửa sổ=[1, 2] -> KHỚP
đã đặt 2/2 nhóm -> True


In [23]:
# Bài 3, 4, 5
def first_match_todo(nums, g, start):
    for j in range(start, len(nums) - len(g) + 1):
        if nums[j:j + len(g)] == g:
            return j
    return -1

def positions(groups, nums):
    res, j = [], 0
    for g in groups:
        p = first_match_todo(nums, g, j)
        if p == -1:
            return None
        res.append(p); j = p + len(g)
    return res

def max_groups(groups, nums):
    j = 0
    for i, g in enumerate(groups):
        p = first_match_todo(nums, g, j)
        if p == -1:
            return i
        j = p + len(g)
    return len(groups)

rng = random.Random(7)
for _ in range(1000):
    nums = [rng.randint(0, 2) for _ in range(rng.randint(1, 12))]
    groups = [[rng.randint(0, 2) for _ in range(rng.randint(1, 2))] for _ in range(rng.randint(1, 4))]
    pos = positions(groups, nums)
    assert (pos is not None) == Solution().canChoose(groups, nums)
    if pos:
        assert all(nums[p:p + len(g)] == g for p, g in zip(pos, groups))
        assert all(pos[k] + len(groups[k]) <= pos[k + 1] for k in range(len(pos) - 1))
    assert max_groups(groups, nums) == max(k for k in range(len(groups) + 1) if can_bruteforce(groups[:k], nums))
print("Bài 3, 4, 5: OK |", positions(*examples[0][:2]), max_groups(*examples[2][:2]))

Bài 3, 4, 5: OK | [3, 6] 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đặt các khối theo thứ tự, sớm nhất có thể":**

```python
i = j = 0
while i < len(groups) and j < len(nums):
    g = groups[i]
    if nums[j:j + len(g)] == g: j += len(g); i += 1
    else:                        j += 1
return i == len(groups)
```

**Checklist:**
- [ ] Khớp thì nhảy `len(g)` (rời nhau), lệch thì tiến `1`?
- [ ] Nhóm sau tìm **sau** nhóm trước (đúng thứ tự)?
- [ ] Trả `i == len(groups)` (đặt đủ mọi nhóm)?
- [ ] Nếu đổi sang chuỗi: có dấu phân cách hai phía?
- [ ] Đã test chồng lấn, sai thứ tự, nhóm lặp, nhóm dài hơn `nums`?